<a href="https://colab.research.google.com/github/financieras/antigen_predictor/blob/main/notebooks/01_dataset_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 01: Exploración del Dataset IEDB

El objetivo de este notebook es entender la estructura del archivo de exportación
completo de IEDB antes de construir nuestro dataset de entrenamiento.

No construimos nada aquí. Solo miramos, filtramos y documentamos lo que encontramos.

Al final del notebook sabremos:
- Qué columnas son relevantes para nuestro proyecto
- Cuántos epítopos hay de SARS-CoV-2 e Influenza A
- Qué proporción de ensayos son positivos vs negativos
- Qué columna identifica la proteína fuente de cada epítopo

---
### Nota sobre el formato del CSV de IEDB

El archivo `epitope_full_v3.csv` tiene **dos filas de cabecera**:
- Fila 0: agrupadores de sección (`Epitope`, `Epitope.1`, `Related Object`...)
- Fila 1: nombres descriptivos reales (`Source Organism`, `Source Molecule`...)

Cargamos el archivo con `header=1` para usar la segunda fila como cabecera
y luego renombramos las columnas clave por posición para evitar ambigüedad
con los nombres duplicados.

## 0. Configuración y montaje de Google Drive

In [1]:
from google.colab import drive
import pandas as pd
import os

drive.mount('/content/drive')

Mounted at /content/drive


## 1. Carga del dataset

Copiamos primero el archivo a almacenamiento local de Colab para acelerar la carga.
Usamos `header=1` para que pandas use la **segunda fila** como nombres de columna,
ignorando la primera fila de agrupadores de sección.

In [2]:
DRIVE_PATH = '/content/drive/MyDrive/epitope/epitope_full_v3.csv'
LOCAL_PATH = '/content/epitope_full_v3.csv'

if not os.path.exists(LOCAL_PATH):
    print("Copiando archivo desde Drive...")
    !cp "{DRIVE_PATH}" "{LOCAL_PATH}"
    print("Copia terminada.")

print("Cargando dataset...")
df = pd.read_csv(LOCAL_PATH, low_memory=False, header=1)
print(f"Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")

Copiando archivo desde Drive...
Copia terminada.
Cargando dataset...
Dataset cargado: 2,421,965 filas × 32 columnas


## 2. Primera inspección: columnas y valores de ejemplo

In [3]:
# Ver todas las columnas con un valor real de ejemplo
print(f"Total de columnas: {len(df.columns)}\n")
for i, col in enumerate(df.columns):
    sample = df[col].dropna().iloc[0] if df[col].notna().any() else "NaN"
    print(f"  {i:>3}. {col:<35} → {str(sample)[:60]}")

Total de columnas: 32

    0. IEDB IRI                            → http://www.iedb.org/epitope/1
    1. Object Type                         → Linear peptide
    2. Name                                → AA + MCM(A1,A2)
    3. Modified Residue(s)                 → A1,A2
    4. Modifications                       → Main chain modification
    5. Starting Position                   → 200.0
    6. Ending Position                     → 201.0
    7. IRI                                 → http://purl.obolibrary.org/obo/CHEBI_34718
    8. Synonyms                            → 1,3-Dinitro-4-chlorobenzene, 1-chloro-2,4-dinitrobenzene, 1-
    9. Source Molecule                     → streptokinase, SKase
   10. Source Molecule IRI                 → http://www.ncbi.nlm.nih.gov/protein/AAB20743.1
   11. Molecule Parent                     → Streptokinase A
   12. Molecule Parent IRI                 → http://www.uniprot.org/uniprot/P10520
   13. Source Organism                     → Streptococcus pyog

## 3. Renombrar columnas clave por posición

El CSV tiene nombres de columna duplicados (varias columnas se llaman igual
porque pertenecen a secciones distintas). Para evitar ambigüedad, renombramos
las columnas que necesitamos usando su posición numérica, identificada
en el diagnóstico previo del archivo.

In [4]:
# Posiciones identificadas en el archivo epitope_full_v3.csv:
#   1  → Name            (secuencia/nombre del epítopo)
#   8  → Source Molecule (nombre de la proteína fuente)
#   9  → Source Molecule IRI (identificador UniProt de la proteína)
#  12  → Source Organism (nombre del organismo fuente)

cols = list(df.columns)
cols[1]  = 'epitope_name'
cols[8]  = 'source_molecule'
cols[9]  = 'source_molecule_iri'
cols[12] = 'source_organism'
df.columns = cols

print("Columnas clave renombradas. Verificación con valores de ejemplo:")
for c in ['epitope_name', 'source_molecule', 'source_molecule_iri', 'source_organism']:
    sample = df[c].dropna().iloc[0] if df[c].notna().any() else "NaN"
    print(f"  {c:<25} → {str(sample)[:70]}")

Columnas clave renombradas. Verificación con valores de ejemplo:
  epitope_name              → Linear peptide
  source_molecule           → 1,3-Dinitro-4-chlorobenzene, 1-chloro-2,4-dinitrobenzene, 1-Chloro-2,4
  source_molecule_iri       → streptokinase, SKase
  source_organism           → http://www.uniprot.org/uniprot/P10520


## 4. Localizar la columna de resultado de ensayo

El archivo de exportación de IEDB incluye datos de epítopos y datos de ensayos.
Necesitamos la columna que indica si el ensayo fue `Positive` o `Negative`.
Como los nombres son genéricos, la buscamos por sus valores.

In [5]:
# Buscar columnas que contengan los valores 'Positive' o 'Negative'
print("Columnas que contienen valores 'Positive' o 'Negative':")
for i, col in enumerate(df.columns):
    vals = df[col].dropna().unique()
    str_vals = [str(v) for v in vals]
    if 'Positive' in str_vals or 'Negative' in str_vals:
        print(f"  Posición {i:>3} | '{col}' → valores únicos: {str_vals[:8]}")

Columnas que contienen valores 'Positive' o 'Negative':


In [6]:
# Renombrar la columna de resultado de ensayo una vez identificada su posición.
# AJUSTA el valor de ASSAY_COL_POSITION con el número que aparece arriba.

ASSAY_COL_POSITION = None  # <-- reemplaza None con el número de posición encontrado

if ASSAY_COL_POSITION is not None:
    cols = list(df.columns)
    cols[ASSAY_COL_POSITION] = 'assay_qualitative'
    df.columns = cols
    print(f"Columna en posición {ASSAY_COL_POSITION} renombrada a 'assay_qualitative'")
    print(f"\nDistribución de valores:")
    print(df['assay_qualitative'].value_counts(dropna=False))
else:
    print("Ajusta ASSAY_COL_POSITION con el número de posición de la celda anterior.")

Ajusta ASSAY_COL_POSITION con el número de posición de la celda anterior.


## 5. Filtrar por organismos de interés

Buscamos filas de SARS-CoV-2 e Influenza A.
Primero miramos los nombres exactos que usa IEDB.

In [7]:
# Top 30 organismos más frecuentes
print("Top 30 organismos más frecuentes en el dataset:")
print(df['source_organism'].value_counts().head(30))

Top 30 organismos más frecuentes en el dataset:
source_organism
http://www.uniprot.org/uniprot/K0BWD0    28876
http://www.uniprot.org/uniprot/P0C6X7    27671
http://www.uniprot.org/uniprot/P27958    15443
http://www.uniprot.org/uniprot/P17763    13826
http://www.uniprot.org/uniprot/P0C6X5    13285
http://www.uniprot.org/uniprot/P0DTD1    11551
http://www.uniprot.org/uniprot/K0BRG7    10957
http://www.uniprot.org/uniprot/P0C6X1    10670
http://www.uniprot.org/uniprot/P59594     9029
http://www.uniprot.org/uniprot/P0DTC2     8700
http://www.uniprot.org/uniprot/P0C6X2     8152
http://www.uniprot.org/uniprot/Q32ZE1     7572
http://www.uniprot.org/uniprot/P0C6U8     5346
http://www.uniprot.org/uniprot/P0DTC1     4974
http://www.uniprot.org/uniprot/Q4DPM2     4834
http://www.uniprot.org/uniprot/K4LC41     4205
http://www.uniprot.org/uniprot/Q4DCJ6     4199
http://www.uniprot.org/uniprot/P15423     4055
http://www.uniprot.org/uniprot/Q6Q1S2     4034
http://www.uniprot.org/uniprot/P03452     3

In [8]:
# Filtrar por SARS-CoV-2 e Influenza A
mask_sars = df['source_organism'].str.contains(
    'SARS-CoV-2|Severe acute respiratory syndrome coronavirus 2',
    case=False, na=False
)
mask_flu = df['source_organism'].str.contains(
    'Influenza A',
    case=False, na=False
)

df_sars = df[mask_sars].copy()
df_flu  = df[mask_flu].copy()

print(f"Filas SARS-CoV-2:  {len(df_sars):>8,}")
print(f"Filas Influenza A: {len(df_flu):>8,}")
print(f"Total combinado:   {len(df_sars) + len(df_flu):>8,}")

Filas SARS-CoV-2:         0
Filas Influenza A:        0
Total combinado:          0


## 6. Distribución de ensayos positivos vs negativos

In [9]:
df_targets = pd.concat([df_sars, df_flu], ignore_index=True)

print("Distribución de resultados de ensayo en nuestros organismos:")
print(df_targets['assay_qualitative'].value_counts(dropna=False))
print(f"\nTotal de filas: {len(df_targets):,}")

Distribución de resultados de ensayo en nuestros organismos:


KeyError: 'assay_qualitative'

In [ ]:
# Solo ensayos positivos
df_positive = df_targets[df_targets['assay_qualitative'] == 'Positive']
print(f"Epítopos con ensayo positivo: {len(df_positive):,}")

n_proteins = df_positive['source_molecule'].nunique()
print(f"Proteínas fuente únicas:      {n_proteins:,}")

## 7. Proteínas más representadas por organismo

Esto nos anticipa qué veremos en la demo final.
Spike debería aparecer arriba en SARS-CoV-2.

In [ ]:
print("Proteínas de SARS-CoV-2 con más epítopos positivos:")
sars_positive = df_sars[df_sars['assay_qualitative'] == 'Positive']
print(sars_positive['source_molecule'].value_counts().head(15))

In [ ]:
print("Proteínas de Influenza A con más epítopos positivos:")
flu_positive = df_flu[df_flu['assay_qualitative'] == 'Positive']
print(flu_positive['source_molecule'].value_counts().head(15))

## 8. Inspección de nulos en columnas clave

In [ ]:
cols_to_check = ['source_organism', 'source_molecule', 'source_molecule_iri', 'assay_qualitative']
print("Valores nulos en columnas clave (subconjunto filtrado):")
print(df_targets[cols_to_check].isnull().sum())
print(f"\nTotal filas: {len(df_targets):,}")

## 9. Guardar el subconjunto filtrado

Guardamos solo las columnas relevantes de nuestros dos organismos.
El Notebook 02 cargará este archivo en lugar del CSV original de 830 MB.

In [ ]:
OUTPUT_PATH = '/content/drive/MyDrive/epitope/iedb_sars_flu_filtered.csv'

cols_to_save = [
    'epitope_name',
    'source_molecule',
    'source_molecule_iri',
    'source_organism',
    'assay_qualitative'
]

df_targets[cols_to_save].to_csv(OUTPUT_PATH, index=False)
print(f"Archivo guardado en: {OUTPUT_PATH}")
print(f"Tamaño: {len(df_targets):,} filas × {len(cols_to_save)} columnas")

## 10. Resumen de hallazgos

Completa esta celda después de ejecutar el notebook con los valores reales.

| Concepto | Valor |
|---|---|
| Total de filas en IEDB | |
| Filas de SARS-CoV-2 | |
| Filas de Influenza A | |
| Epítopos positivos (combinado) | |
| Proteínas únicas con epítopos positivos | |
| Proteína más representada en SARS-CoV-2 | |
| Proteína más representada en Influenza A | |
| Posición de columna `assay_qualitative` | |

### Columnas clave confirmadas
| Nombre en nuestro código | Posición CSV | Nombre original en IEDB |
|---|---|---|
| `epitope_name` | 1 | Name |
| `source_molecule` | 8 | Source Molecule |
| `source_molecule_iri` | 9 | Source Molecule IRI |
| `source_organism` | 12 | Source Organism |
| `assay_qualitative` | ? | (a confirmar) |

### Decisiones para el Notebook 02
- `label = 1` → proteínas con al menos un epítopo con `assay_qualitative == 'Positive'`
- `label = 0` → proteínas del mismo patógeno sin epítopos positivos
- Archivo de entrada para NB02: `iedb_sars_flu_filtered.csv`